In [ ]:
import pandas as pd
fp = "../data/sba_loans_prepared/sba_loans_stage1.csv"
df = pd.read_csv(fp)

In [ ]:
df

In [ ]:
df[df.LoanStatus.isna()]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
target_recode = {"PIF": 0, "CHGOFF": 1}
df["LoanStatus"] = df["LoanStatus"].replace(target_recode)

In [ ]:
cols = df.columns.to_list()

In [ ]:
NUM_COLS = ['LoanStatus', 'GrossChargeOffAmount', 'NumPmtsMade',  'GrossApproval']
data_types = {k: 'category' for k in cols if k not in NUM_COLS}

In [ ]:
ID_COLS = ["BorrName", "BankFDICNumber", "LoanID"]
df_id = df[ID_COLS]

In [ ]:
df = df.astype(data_types)

In [ ]:
EXCLUDE_COLS = NUM_COLS + ID_COLS
cat_cols = [c for c in cols if c not in EXCLUDE_COLS]
cat_cols = [c for c in cols if c not in EXCLUDE_COLS]

In [ ]:
cat_cols

In [ ]:
from category_encoders import *

In [ ]:
X = df[cat_cols]
Y = df["LoanStatus"]

In [ ]:
NUM_PREDS = [c for c in NUM_COLS if c not in["LoanStatus", "GrossChargeOffAmount"]]

In [ ]:
NUM_PREDS

In [ ]:
df_train, df_test, y_train, y_test = train_test_split(X, Y, test_size=0.15, random_state=42)

In [ ]:
train_num_ind = df_train.index
df_train_num = df[df.index.isin(train_num_ind)][NUM_PREDS]
test_num_ind = df_test.index
df_test_num = df[df.index.isin(test_num_ind)][NUM_PREDS]

In [ ]:
enc = TargetEncoder(cols=cat_cols, min_samples_leaf=20, smoothing=10).fit(df_train, y_train)

In [ ]:
enc_dataset_train = enc.transform(df_train)
enc_dataset_test = enc.transform(df_test)

In [ ]:
df_train = pd.concat([enc_dataset_train,df_train_num], axis = 1)
df_test = pd.concat([enc_dataset_test,df_test_num], axis = 1)

In [ ]:
df_test.shape

In [ ]:
df_train, df_val, y_train, y_val = train_test_split(df_train, y_train, test_size=0.2, random_state=42)

In [ ]:
y_train

In [ ]:
df_train["LoanStatus"] = y_train
df_val["LoanStatus"] = y_val
df_test["LoanStatus"] = y_test

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
preds = [ c for c in df_train.columns.tolist() if c != "LoanStatus"]
X_train = df_train[preds]
scaler.fit(X_train)
X_train_std = scaler.transform(X_train)
df_train = pd.DataFrame(X_train_std)
df_train.columns = preds
df_train.loc[:, "LoanStatus"] = y_train.values

In [ ]:
df_train.shape

In [ ]:
X_test = df_test[preds]
X_test_std = scaler.transform(X_test)
df_test = pd.DataFrame(X_test_std, columns = preds)
df_test.loc[:, "LoanStatus"] = y_test.values

In [ ]:
X_val = df_val[preds]
X_val_std = scaler.transform(X_val)
df_val = pd.DataFrame(X_val_std, columns = preds)
df_val.loc[:, "LoanStatus"] = y_val.values

In [ ]:
train_ids = df_id.index.isin(df_train.index)
test_ids = df_id.index.isin(df_test.index)
val_ids = df_id.index.isin(df_val.index)

In [ ]:
df_train_lookup = df_id[train_ids]
df_test_lookup = df_id[test_ids]
df_val_lookup = df_id[val_ids]

In [ ]:
fp =  "../data/sba_loans_prepared/sba_loans_train_id_lookup.csv"
df_train_lookup.to_csv(fp, index=False)
fp =  "../data/sba_loans_prepared/sba_loans_test_id_lookup.csv"
df_test_lookup.to_csv(fp, index=False)
fp =  "../data/sba_loans_prepared/sba_loans_val_id_lookup.csv"
df_val_lookup.to_csv(fp, index=False)


In [ ]:
df_train

In [ ]:
fp =  "../data/sba_loans_prepared/sba_loans_num_enc_train.csv"
df_train.to_csv(fp, index=False)

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_test.csv"
df_test.to_csv(fp, index=False)

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_val.csv"
df_val.to_csv(fp, index=False)

In [ ]:
df_test

In [ ]:
BORR_INFO = ["BorrName", "BorrCity", "BorrState", "BorrZip"]

In [ ]:
train_borr_info_df = df[train_ids][BORR_INFO]
test_borr_info_df = df[test_ids][BORR_INFO]
val_borr_info_df = df[val_ids][BORR_INFO]

In [ ]:
fp_train = "../data/sba_loans_prepared/sba_train_borr_info.csv"
train_borr_info_df.to_csv(fp_train, index=True)
fp_val = "../data/sba_loans_prepared/sba_val_borr_info.csv"
val_borr_info_df.to_csv(fp_val, index=True)
fp_test = "../data/sba_loans_prepared/sba_val_borr_info.csv"
val_borr_info_df.to_csv(fp_val, index=True)